In [0]:
##aws_utils
import boto3
from botocore.exceptions import NoCredentialsError

from pyspark.dbutils import DBUtils
from pyspark.sql import SparkSession
 
spark = SparkSession.builder.getOrCreate()
dbutils = DBUtils(spark)

# Function to get AWS credentials from Databricks secrets
def get_aws_credentials(secrets_scope, key_name):
    try:
        # Databricks secrets configuration
   
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        # if aws_secret_key is not None:
        #     print("AWS secret key retrieved successfully.")
        # else:
        #     print("AWS secret key is None. Please check Databricks secrets configuration.")
 
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None

# Function to get AWS credentials from Databricks secrets
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration
 
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
   
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

# Function to connect to AWS and access SSM parameter
def connect_to_aws_and_access_ssm(ssm_key, ssm_client, aws_secret_key):
 
    if aws_secret_key is None:
        print("AWS secret key not available.")
        return
 
    # Access SSM parameter /cfn/a360/reports_output_path
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )
 
    return response['Parameter']['Value']

# Function to get the environment prefix
def get_env_prefix():
    secrets_scope = "aws-databricks"
    aws_keys = get_scope_key_names(secrets_scope)
    aws_key = aws_keys[0] if aws_keys else None
    # Get AWS secret key from Databricks secrets
    aws_secret_key = get_aws_credentials(secrets_scope,aws_key)
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)
 
    S3_client = boto3.client('s3', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)  

    env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix', ssm_client, aws_secret_key)
    return env_prefix

env = get_env_prefix()
print(f"Env is: {env}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "Europe/Athens")


def is_qualifying_source_row():
    return (
        (F.col("BusinessDate") >= '2026-08-05')
        & ((F.col("assignee_team").isNull()) | (F.col("assignee_team") != "PRE_FIRST_CALL"))
        & (F.col("_include_in_output_tables") == True)
        & (F.col("ApprovalStatus").isin("LVL_2_APPROVED", "AUTO_APPROVED"))
        & (F.col("AssignmentStatus") == "approved")
        & (~F.col("PortfolioName").isin(
            "MONZA PORTFOLIO - RETAIL",
            "IMOLA PORTFOLIO - RETAIL",
            "ASTRAEA PORTFOLIO RETAIL"
        ))
    )


def resolve_account_channel():
    # --- Spec §6 & §8.1 Channel Rule (Segmentation table) ----------------------
    # Returns a Column aliased ACCOUNT_ASSIGNMENT_CHANNEL. The outer coalesce maps
    # a NULL result to 'No Channel' (test case: "Null channel -> Replace with
    # 'No Channel'"). Extracted so production and the BRD tests share one source.
    return F.coalesce(
        F.when(
            # Case requested in ticket PMT-1167
            F.col("KeepOrReassign") == "not-assigned",
            F.lit("No Channel")
        ).when(
            F.col("assignee_channel").isin(
                "Outsourcing Collection", "Legal Agencies", "Legal Agencies 3869"
            ),
            F.col("assignee_team")
        ).otherwise(F.col("assignee_channel")),
        F.lit("No Channel"),
    ).alias("ACCOUNT_ASSIGNMENT_CHANNEL")


def resolve_customer_channel():
    # --- Spec §6 Channel Rule (CustomerSegment table): NULL -> 'No Channel'. ----
    return F.coalesce(F.col("CUSTOMER_CHANNEL"), F.lit("No Channel")).alias("CUSTOMER_CHANNEL")


def load_data(table: str):
    df = spark.read.table(table)
    df_filtered = df.filter(is_qualifying_source_row())

    return df_filtered


def write_data(df, table):
    df.write.mode("append").option("mergeSchema", "true").saveAsTable(table)


def append_with_carry_forward(new_df, table):
    # --- (carry-forward requirement) ---------------------------------------
    # Append `new_df` (already stamped with this run's BusinessDate/IngestionTime)
    # AND re-append the table's CURRENT max-BusinessDate rows, re-stamped with the
    # SAME run BusinessDate/IngestionTime. The existing snapshot is read BEFORE the
    # new data is written, so "max BusinessDate" refers to the PREVIOUS run.
    #
    # NOTE: rows are NOT de-duplicated between the new data and the carried-forward
    # snapshot. If a key (e.g. ACCOUNT_NUMBER / CDI) appears in BOTH, it ends up
    # twice for this BusinessDate. To keep only the new version instead, left-anti
    # the carry_forward against new_df on the key before the union.
    full = f"{catalog}.{schema}.{table}"

    new_df = new_df.cache()

    existing = spark.read.table(full)
    max_bd = existing.agg(F.max("BusinessDate")).first()[0]
    if max_bd is None:
        # First run / empty table -> nothing to carry forward.
        carry_forward = existing.limit(0)
    else:
        carry_forward = (
            existing.filter(F.col("BusinessDate") == max_bd)
            .withColumn("BusinessDate", F.lit(run_business_date))
            .withColumn("IngestionTime", F.lit(run_ingestion_time))
        )

    # Single append of the new rows + the carried-forward snapshot (matched by name).
    combined = new_df.unionByName(carry_forward, allowMissingColumns=True)
    write_data(combined, table=full)
    return new_df.count(), carry_forward.count()


def filter_data(df, partition_by: str):
    # --- CHANGED (Spec §3 Latest Record Selection + §4 Deterministic dedup) ----
    # Per partition key:
    #   1) latest business date     -> max(BusinessDate)
    #   2) within it, latest update -> max(_update_timestamp)
    # Previously the window ordered by _update_timestamp ONLY (BusinessDate was
    # ignored). dataset_hash is the final deterministic tie-breaker so the kept
    # ("first") record is reproducible across runs (§4 / §9 test cases:
    # "Same timestamp, different values -> Keep first by dataset_hash",
    # "Multiple dataset_hash -> Deterministic").
    window_spec = Window.partitionBy(partition_by).orderBy(
        F.col("BusinessDate").desc(),
        F.col("_update_timestamp").desc(),
        F.col("dataset_hash").asc(),
    )
    df_with_rownum = df.withColumn("row_num", F.row_number().over(window_spec))
    latest_rows = df_with_rownum.filter(F.col("row_num") == 1).drop("row_num")
    return latest_rows


catalog = "`p-cdh-delivery`"
schema = "`pmt-assignments`"

# --- freeze ONE BusinessDate + IngestionTime for the whole run, so the new
# data AND the carried-forward data are stamped with IDENTICAL values (and
# current_timestamp() cannot drift between separate writes).
_run_stamp = spark.sql(
    "SELECT date_format(current_date(), 'yyyyMMdd') AS bd, current_timestamp() AS it"
).first()
run_business_date = _run_stamp["bd"]
run_ingestion_time = _run_stamp["it"]

business_date_col = F.lit(run_business_date).alias("BusinessDate")
ingestion_time_col = F.lit(run_ingestion_time).alias("IngestionTime")

# --- control/tracking table that stores which result sets (dataset_hash
# values) have already been parsed, so we only process newly-arrived results.
tracking_table = "assignmentsoffercollections_parsed_datasets"

In [0]:
def get_jdbc_connection_properties():
    """
    Utility function to create JDBC connection properties.
    Centralizes database credentials and connection configuration.
    
    Returns:
        tuple: (jdbc_url, connection_properties)
    """
    jdbc_hostname = "dnaappsprod.cluster-cnf8phxc0hd5.eu-central-1.rds.amazonaws.com"
    jdbc_port = 5432
    jdbc_database = "pt_prod"
    jdbc_url = f"jdbc:postgresql://{jdbc_hostname}:{jdbc_port}/{jdbc_database}"
 
    connection_properties = {
        "user": dbutils.secrets.get(scope="pmtool", key="db-user"),
        "password": dbutils.secrets.get(scope="pmtool", key="db-password"),
    }
 
    return jdbc_url, connection_properties

def read_table_from_db(table_name):
    """
    Generic utility function to read a table from the database via JDBC.
    
    Args:
        table_name (str): Fully qualified table name (e.g., "public.actors_app_actor")
    
    Returns:
        DataFrame: Spark DataFrame containing the table data
    """
    jdbc_url, connection_properties = get_jdbc_connection_properties()
    df = spark.read.jdbc(
        url=jdbc_url,
        table=table_name,
        properties=connection_properties
    )
    return df

In [0]:
def get_pm_tool_actors_df():
    """Fetch PM tool actors data from database."""
    return read_table_from_db("public.actors_app_actor")

def get_qualco_mappings():
    """Fetch Qualco mappings data from database."""
    return read_table_from_db("public.assignment_app_qualcomapping")

In [0]:
df = load_data(table='`p-ptools`.assignments.offer_collection')

# --- (Spec §6 Update frequency, via dataset_hash tracking) ---------------
# Each result set in offer_collection carries a dataset_hash shared by all of its
# rows, identifying one PMTool execution. We keep a control table of already-
# parsed dataset_hashes; any dataset_hash present in the source but absent from
# that table is a NEW result.
#
#   * segmentation & customersegment -> append ONLY the rows of new result sets
#     (incremental; history preserved).
#   * personalassignment             -> rebuilt DAILY from all current data,
#     regardless of whether new results arrived.

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.{schema}.{tracking_table} (
        dataset_hash STRING,
        parsed_date STRING,
        parsed_timestamp TIMESTAMP
    )
    USING DELTA
""")

# dataset_hashes present in the current (filtered) source vs. those already parsed
source_hashes = df.select("dataset_hash").distinct()

parsed_hashes = (
    spark.read.table(f"{catalog}.{schema}.{tracking_table}")
    .selectExpr("dataset_hash as parsed_hash")
    .distinct()
)

new_hashes_df = source_hashes.join(
    parsed_hashes, 
    source_hashes.dataset_hash == parsed_hashes.parsed_hash, 
    how="left_anti"
)

new_hash_count = new_hashes_df.count()
has_new_results = new_hash_count > 0

new_hash_list = [r["dataset_hash"] for r in new_hashes_df.collect()]

print(f"Found {new_hash_count} new dataset_hash(es) to parse. has_new_results={has_new_results}")
if not has_new_results:
    print(
        "No new PMTool results -> Segmentation & CustomerSegment will be skipped; "
        "PersonalAssignment will still run for today."
    )

In [0]:
df_dexposure = spark.read.table("`p-cdh`.derivative.dexposure")

max_bd = df_dexposure.agg(F.max("BusinessDate")).first()[0]

df_dexposure_filtered = df_dexposure.filter(
    (F.col("BusinessDate") == max_bd)
    & (F.col("IsActiveFlag") == True)
    & (
        (
            (F.col("PortfolioName") != "Core Retail")
        ) | (
            (F.col("PortfolioName") == "Core Retail") & (F.col("ONBALANCE") > 0)
        )
    )
    & (~F.col("PortfolioName").isin(
        "MONZA PORTFOLIO - RETAIL",
        "IMOLA PORTFOLIO - RETAIL",
        "ASTRAEA PORTFOLIO RETAIL"
    ))
)

active_exposure_keys = df_dexposure_filtered.select("ExposureKeyQCAlpha").distinct()
df_active = df.join(
    F.broadcast(active_exposure_keys), 
    df["ExposureKey"] == active_exposure_keys["ExposureKeyQCAlpha"], 
    how="left_semi"
).cache()

In [0]:
# --- (Spec §7 Data Validation - Portfolio Coverage Check) ----------------
# Ensure every active exposure is represented in the current data. Compares the
# distinct ExposureKey count surviving all source filters (df_active) against the
# distinct active ExposureKey count from dexposure. A mismatch means some active
# exposures have no qualifying approved assignment -> emit a notification warning
# (does NOT stop the run).
output_exposures = df_active.select("ExposureKey").distinct().count()
dexposure_exposures = active_exposure_keys.count()

if output_exposures != dexposure_exposures:
    print(
        f"WARNING [Portfolio Coverage]: distinct ExposureKey in output "
        f"({output_exposures}) != distinct active ExposureKey in dexposure "
        f"({dexposure_exposures}). "
        f"{dexposure_exposures - output_exposures} active exposure(s) "
        f"are not represented in the current data."
    )
else:
    print(f"Portfolio coverage OK: {output_exposures} active exposures fully covered.")

In [0]:
try:
    qualco_mappings_df = get_qualco_mappings()
    qualco_mapping_rows = [] if qualco_mappings_df.isEmpty() else [r.asDict() for r in qualco_mappings_df.collect()]
except Exception:
    qualco_mapping_rows = []

In [0]:
table_name = "assignmentsoffercollectionssegmentation"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.{schema}.{table_name} (
        ACCOUNT_NUMBER VARCHAR(40),
        ACCOUNT_SEGMENT VARCHAR(50),
        ACCOUNT_SEGMENT_REF_DATE STRING,
        ACCOUNT_SEGMENT_PROPOSAL VARCHAR(50),
        ACCOUNT_SEGMENT_AMOUNT DECIMAL(19,4),
        ACCOUNT_EXTRA_SEGMENT VARCHAR(50),
        ACCOUNT_SUB_SEGMENT VARCHAR(50),
        ACCOUNT_ASSIGNMENT_CHANNEL VARCHAR(50),
        ACCOUNT_BALANCE_CATEGORY VARCHAR(50),
        dataset_hash STRING,
        BusinessDate STRING,
        IngestionTime TIMESTAMP
    )
    USING DELTA
""")

# Always write data, regardless of has_new_results

print(f"[segmentation] df_active={df_active.count()} rows")

input_df = filter_data(
    df_active.filter(F.col("ExposureKey").isNotNull()),
    partition_by="ExposureKey",
)

input_df_columns = input_df.columns

# --- Custom Qualco mapping override -----------------------------------------
# qualco_mappings_df holds one row per
# PortfolioName that needs a CUSTOM mapping. Every column other than id /
# PortfolioName holds the NAME of the column in `input_df` that should feed the
# corresponding output column, for rows belonging to that PortfolioName.
#
# For a PortfolioName found in qualco_mappings_df, the value is taken straight
# from that named source column (bypassing the fixed/default expression below).
# Every other PortfolioName keeps using the fixed expression, unchanged.

def resolve_mapped_source(source_df, target_col, default_expr):
    """
    Column pulled from a portfolio-specific source column when qualco_mappings_df
    has a custom mapping for (PortfolioName, target_col); otherwise `default_expr`.
    """
    expr = default_expr
    for row in qualco_mapping_rows:
        portfolio = row.get("PortfolioName")
        mapped_col = row.get(target_col)
        if portfolio and mapped_col and mapped_col in input_df_columns:
            expr = F.when(F.col("PortfolioName") == portfolio, F.col(mapped_col)).otherwise(expr)
        elif portfolio and mapped_col:
            print(f"WARNING: qualco mapping for PortfolioName='{portfolio}' points "
                  f"target='{target_col}' to source column '{mapped_col}', which is "
                  f"not present in the input data -> keeping fixed mapping for it.")
    return expr

def to_amount(col):
    return F.when(col.isNotNull() & (col != "-"), col.cast("decimal(19,4)")).otherwise(None)

segmentation_df = input_df.select(
    resolve_mapped_source(input_df, "ACCOUNT_NUMBER", F.col("ExposureKey")).alias("ACCOUNT_NUMBER"),
    resolve_mapped_source(
        input_df, 
        "ACCOUNT_SEGMENT", 
        F.col("segment_mapped_values")).alias("ACCOUNT_SEGMENT"),
        F.date_format(
            resolve_mapped_source(input_df, "ACCOUNT_SEGMENT_REF_DATE", F.col("BusinessDate")),
            "yyyyMMdd"
        ).alias("ACCOUNT_SEGMENT_REF_DATE"
    ),
    resolve_mapped_source(
        input_df, 
        "ACCOUNT_SEGMENT_PROPOSAL", 
        F.col("ACCOUNT_SEGMENT_PROPOSAL")).alias("ACCOUNT_SEGMENT_PROPOSAL"),
        to_amount(
            resolve_mapped_source(input_df, "ACCOUNT_SEGMENT_AMOUNT", F.col("ACCOUNT_SEGMENT_AMOUNT"))
        ).alias("ACCOUNT_SEGMENT_AMOUNT"),
        F.substring(
            resolve_mapped_source(input_df, "ACCOUNT_EXTRA_SEGMENT", F.col("Segment")), 1, 50
        ).alias("ACCOUNT_EXTRA_SEGMENT"
    ),
    resolve_mapped_source(input_df, "ACCOUNT_SUB_SEGMENT", F.col("ACCOUNT_SUB_SEGMENT")).alias("ACCOUNT_SUB_SEGMENT"),
    resolve_mapped_source(input_df, "ACCOUNT_ASSIGNMENT_CHANNEL", resolve_account_channel()).alias("ACCOUNT_ASSIGNMENT_CHANNEL"),
    resolve_mapped_source(input_df, "ACCOUNT_BALANCE_CATEGORY", F.col("ACCOUNT_BALANCE_CATEGORY")).alias("ACCOUNT_BALANCE_CATEGORY"),
    F.col("dataset_hash"),
    business_date_col,
    ingestion_time_col,
    F.col("PortfolioName"),
    F.col("_update_timestamp")
)

# Load and align qca_thirdpartiesaccountsegmentation
qca_df = spark.read.table("`p-de-sandbox`.`dna-ba`.qca_thirdpartiesaccountsegmentation")
qca_df_aligned = qca_df.select(
    F.col("ACCOUNT_NUMBER"),
    F.col("ACCOUNT_SEGMENT"),
    F.col("ACCOUNT_SEGMENT_REF_DATE"),
    F.col("ACCOUNT_SEGMENT_PROPOSAL"),
    F.col("ACCOUNT_SEGMENT_AMOUNT").cast("decimal(19,4)").alias("ACCOUNT_SEGMENT_AMOUNT"),
    F.col("ACCOUNT_EXTRA_SEGMENT"),
    F.col("ACCOUNT_SUB_SEGMENT"),
    F.col("ACCOUNT_ASSIGNMENT_CHANNEL"),
    F.col("ACCOUNT_BALANCE_CATEGORY"),
    F.col("DatasetHash").alias("dataset_hash"),
    business_date_col,
    ingestion_time_col
)

# Union the two DataFrames, aligning columns
segmentation_df_final = segmentation_df.select(
    "ACCOUNT_NUMBER",
    "ACCOUNT_SEGMENT",
    "ACCOUNT_SEGMENT_REF_DATE",
    "ACCOUNT_SEGMENT_PROPOSAL",
    "ACCOUNT_SEGMENT_AMOUNT",
    "ACCOUNT_EXTRA_SEGMENT",
    "ACCOUNT_SUB_SEGMENT",
    "ACCOUNT_ASSIGNMENT_CHANNEL",
    "ACCOUNT_BALANCE_CATEGORY",
    "dataset_hash",
    "BusinessDate",
    "IngestionTime"
).unionByName(qca_df_aligned, allowMissingColumns=True)

# Keep a new df with data from `p-ptools`.strategy.FLDTfeedingSegmentation on the most recent _creation_timestamp col value
fldt_df = spark.read.table("`p-ptools`.strategy.FLDTfeedingSegmentation")
max_creation_ts = fldt_df.agg(F.max("_creation_timestamp")).first()[0]
fldt_latest_df = (
    fldt_df.filter(F.col("_creation_timestamp") == max_creation_ts)
    .dropDuplicates(["ExposureKey"])  # FLDT can have duplicate ExposureKeys at the same _creation_timestamp
    .select(
        F.col("ExposureKey").alias("FLDT_ExposureKey"),
        F.col("Segment").alias("FLDT_Segment")
    )
)

# Only update ACCOUNT_EXTRA_SEGMENT for rows with BusinessDate == run_business_date
latest_bd_df = segmentation_df_final.filter(F.col("BusinessDate") == run_business_date)
other_bd_df = segmentation_df_final.filter(F.col("BusinessDate") != run_business_date)

latest_bd_df = latest_bd_df.join(
    fldt_latest_df,
    latest_bd_df.ACCOUNT_NUMBER == fldt_latest_df.FLDT_ExposureKey,
    how="left"
).withColumn(
    "ACCOUNT_EXTRA_SEGMENT",
    F.when(F.col("FLDT_Segment").isNotNull(), F.substring(F.col("FLDT_Segment"), 1, 50))
     .otherwise(F.col("ACCOUNT_EXTRA_SEGMENT"))
).drop("FLDT_ExposureKey", "FLDT_Segment")

# Combine updated latest_bd_df with other_bd_df
segmentation_df_final_updated = latest_bd_df.unionByName(other_bd_df, allowMissingColumns=True)

write_data(segmentation_df_final_updated, table=f'{catalog}.{schema}.{table_name}')
print(f"Wrote {segmentation_df_final_updated.count()} lines (PMTool + QCA, FLDT Segment update) to {catalog}.{schema}.{table_name}")

In [0]:
table_name = "assignmentsoffercollectionscustomersegment"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.{schema}.{table_name} (
        CDI VARCHAR(50),
        CUSTOMER_SEGMENT VARCHAR(50),
        CUSTOMER_EXTRA_SEGMENT VARCHAR(50),
        CUSTOMER_SUB_SEGMENT VARCHAR(50),
        CUSTOMER_BALANCE_CATEGORY VARCHAR(50),
        CUSTOMER_CHANNEL VARCHAR(50),
        dataset_hash STRING,
        BusinessDate STRING,
        IngestionTime TIMESTAMP
    )
    USING DELTA
""")

# --- write ONLY when new results arrived, and INCREMENTALLY -
if not has_new_results:
    print(f"Skipping {table_name}: no new PMTool results received.")
else:
    # Restrict to NEW result sets (§6), require CDI NOT NULL (§8.2). filter_data
    # already keeps exactly one deterministic row per CDI (partition by CDI,
    # ordered by BusinessDate / _update_timestamp / dataset_hash)
    new_active = df_active #.filter(F.col("dataset_hash").isin(new_hash_list))
    print(f"[customersegment] df_active={df_active.count()} rows | "
          f"new_active={new_active.count()} rows")
    input_df = filter_data(
        new_active.filter(F.col("CDI").isNotNull()),
        partition_by="CDI",
    )

    # --- Custom Qualco mapping override -------------------------------------
    # Reuses qualco_mapping_rows / resolve_mapped_source unchanged: 
    # for any PortfolioName present in qualco_mappings_df, each output 
    # column is populated from the source column named in that mapping 
    # instead of the fixed default below.
    customer_segment_df = input_df.select(
        resolve_mapped_source(input_df, "CDI", F.col("CDI")).alias("CDI"),
        resolve_mapped_source(input_df, "CUSTOMER_SEGMENT", F.col("CUSTOMER_SEGMENT")).alias("CUSTOMER_SEGMENT"),
        resolve_mapped_source(input_df, "CUSTOMER_EXTRA_SEGMENT", F.col("CUSTOMER_EXTRA_SEGMENT")).alias("CUSTOMER_EXTRA_SEGMENT"),
        resolve_mapped_source(input_df, "CUSTOMER_SUB_SEGMENT", F.col("CUSTOMER_SUB_SEGMENT")).alias("CUSTOMER_SUB_SEGMENT"),
        resolve_mapped_source(input_df, "CUSTOMER_BALANCE_CATEGORY", F.col("CUSTOMER_BALANCE_CATEGORY")).alias("CUSTOMER_BALANCE_CATEGORY"),
        resolve_mapped_source(input_df, "CUSTOMER_CHANNEL", resolve_customer_channel()).alias("CUSTOMER_CHANNEL"),
        F.col("dataset_hash"),
        business_date_col,
        ingestion_time_col,
        F.col("PortfolioName"),
        F.col("_update_timestamp")
    )

    # Load and align qca_thirdpartiescustomersegmentation
    qca_cust_df = spark.read.table("`p-de-sandbox`.`dna-ba`.qca_thirdpartiescustomersegmentation")
    qca_cust_df_aligned = qca_cust_df.select(
        F.col("CDI"),
        F.col("CUSTOMER_SEGMENT"),
        F.col("CUSTOMER_EXTRA_SEGMENT"),
        F.col("CUSTOMER_SUB_SEGMENT"),
        F.col("CUSTOMER_BALANCE_CATEGORY"),
        F.col("CUSTOMER_CHANNEL"),
        F.col("DatasetHash").alias("dataset_hash"),
        business_date_col,
        ingestion_time_col
    )

    # Union the two DataFrames, aligning columns
    customer_segment_df_final = customer_segment_df.select(
        "CDI",
        "CUSTOMER_SEGMENT",
        "CUSTOMER_EXTRA_SEGMENT",
        "CUSTOMER_SUB_SEGMENT",
        "CUSTOMER_BALANCE_CATEGORY",
        "CUSTOMER_CHANNEL",
        "dataset_hash",
        "BusinessDate",
        "IngestionTime"
    ).unionByName(qca_cust_df_aligned, allowMissingColumns=True)

    write_data(customer_segment_df_final, table=f'{catalog}.{schema}.{table_name}')
    print(f"Wrote {customer_segment_df_final.count()} lines (PMTool + QCA) to {catalog}.{schema}.{table_name}")
    # print("Preview:")
    # display(customer_segment_df_final.limit(100))

In [0]:
# --- record the result sets we just parsed so they are not reprocessed ---
# Runs only when there were new results, AFTER segmentation & customersegment have
# been written. (personalassignment below is a daily snapshot and does not affect
# tracking.) Each new dataset_hash is stored with the date/timestamp it was parsed.
if has_new_results:
    parsed_now = new_hashes_df.select(
        "dataset_hash",
        F.date_format(F.current_date(), "yyyyMMdd").alias("parsed_date"),
        F.current_timestamp().alias("parsed_timestamp"),
    )
    parsed_now.write.mode("append").saveAsTable(f"{catalog}.{schema}.{tracking_table}")
    print(f"Marked {new_hash_count} dataset_hash(es) as parsed in {catalog}.{schema}.{tracking_table}")
else:
    print("No new dataset_hashes to record in the tracking table.")

In [0]:
table_name = "assignmentsoffercollectionspersonalassignment"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.{schema}.{table_name} (
        ACCOUNT_NUMBER VARCHAR(40),
        User VARCHAR(40),
        dataset_hash STRING,
        BusinessDate STRING,
        IngestionTime TIMESTAMP
    )
    USING DELTA
""")

# 1. Keep a temporary df with all the data (on max businessdate) from personalassignment table (before saving new data)
personalassignment_full = spark.read.table(f"{catalog}.{schema}.{table_name}")

if personalassignment_full.count() > 0:
    max_bd_before = personalassignment_full.agg(F.max("BusinessDate")).first()[0]
    temp_before_df = personalassignment_full.filter(F.col("BusinessDate") == max_bd_before)
else:
    temp_before_df = personalassignment_full.limit(0)

actors_df = get_pm_tool_actors_df() \
    .select("actor", "username") \
    .distinct()

# NOTE: this table is NOT gated on has_new_results — it runs every day and
# re-writes ALL current qualifying data (one row per active ExposureKey) with the
# current BusinessDate and a new IngestionTime.
input_df = filter_data(
    df_active.filter(F.col("ExposureKey").isNotNull()),
    partition_by="ExposureKey",
)

personal_assignment_df = input_df.select(
    F.col("ExposureKey").alias("ACCOUNT_NUMBER"),
    "assignee_actor",
    F.col("dataset_hash"),
    business_date_col,
    ingestion_time_col,
    F.col("PortfolioName"),
    F.col("_update_timestamp")
)

personal_assignment_df = personal_assignment_df.join(
    F.broadcast(actors_df),
    personal_assignment_df.assignee_actor == actors_df.actor,
    how="left"
)

# Cache: personal_assignment_df is reused (union + count)
personal_assignment_df = personal_assignment_df.withColumnRenamed("username", "User").drop("assignee_actor", "actor").cache()

# ========= Build the adhoc rows (appended together with the PMTool snapshot) =========
# Adhoc rows share this run's frozen BusinessDate & IngestionTime. 
# dataset_hash is kept if present, otherwise tagged 'ADHOC' (adhoc data is not a PMTool result set).
df_adhoc = spark.sql("""
    SELECT *
    FROM `p-de-sandbox`.`dna-ba`.paadhoc
""")

if "dataset_hash" not in df_adhoc.columns:
    df_adhoc = df_adhoc.withColumn("dataset_hash", F.lit("ADHOC"))

df_adhoc = df_adhoc.withColumn("BusinessDate", business_date_col)
df_adhoc = df_adhoc.withColumn("IngestionTime", ingestion_time_col)

# Align adhoc to the target schema (column set + order) before appending.
df_adhoc = df_adhoc.select("ACCOUNT_NUMBER", "User", "dataset_hash", "BusinessDate", "IngestionTime")

# This run's data = PMTool snapshot + adhoc rows.
new_personal_df = personal_assignment_df.unionByName(df_adhoc, allowMissingColumns=True)



# --- NO carry-forward for personalassignment. -----------------------
# Unlike segmentation & customersegment (which append ONLY new result sets and so
# use carry-forward to keep a complete snapshot), personalassignment is NOT gated
# on has_new_results: it already re-writes ALL current data every run with the
# current BusinessDate and a new IngestionTime. Re-appending the previous
# max-BusinessDate snapshot on top of that would duplicate rows, so we append the
# new snapshot directly.
write_data(new_personal_df, table=f'{catalog}.{schema}.{table_name}')
print(f"Wrote {new_personal_df.count()} lines (PMTool + adhoc) to {catalog}.{schema}.{table_name}")

# 2. Keep another temporary df of the data from personalassignment (again on max businessDate)
personalassignment_full_after = spark.read.table(f"{catalog}.{schema}.{table_name}")
max_bd_after = personalassignment_full_after.agg(F.max("BusinessDate")).first()[0]
temp_after_df = personalassignment_full_after.filter(F.col("BusinessDate") == max_bd_after)

# 3. Find which records are missing from the second temp table compared to the first temp table and keep the delta
#    (i.e., records in temp_before_df not present in temp_after_df)
delta_df = temp_before_df.join(
    temp_after_df.select("ACCOUNT_NUMBER"),
    on="ACCOUNT_NUMBER",
    how="left_anti"
)

# 4. Compare the delta to the dexposure table and find which records are active
max_bd = df_dexposure.agg(F.max("BusinessDate")).first()[0]

df_dexposure_filtered = df_dexposure.filter(
    (F.col("BusinessDate") == max_bd)
    & (F.col("IsActiveFlag") == True)
    & (F.col("PortfolioName").isin(
        'Mercury', 
        'Orbit', 
        'Light', 
        'Cell', 
        'Pearl Portfolio A'
    ))
)

active_exposure_keys = df_dexposure_filtered.select("ExposureKeyQCAlpha").distinct()
active_delta_df = delta_df.join(
    F.broadcast(active_exposure_keys),
    delta_df.ACCOUNT_NUMBER == active_exposure_keys.ExposureKeyQCAlpha,
    how="inner"
).drop("ExposureKeyQCAlpha")

active_delta_count = active_delta_df.count()

# 5. Write back the active delta records with the new BusinessDate and IngestionTime
if active_delta_count > 0:
    active_delta_df = active_delta_df.withColumn("BusinessDate", business_date_col)
    active_delta_df = active_delta_df.withColumn("IngestionTime", ingestion_time_col)
    write_data(active_delta_df, table=f'{catalog}.{schema}.{table_name}')
    print(f"Wrote {active_delta_count} active delta lines to {catalog}.{schema}.{table_name}")
else:
    print("No active delta records to write back.")

In [0]:
spark.sql(f'''
with max_dt as (
  select max(businessDate) as businessdate
  FROM `{env}-cdh`.landed.qca_personal_assignment_8702
)

MERGE INTO `{env}-cdh-delivery`.`pmt-assignments`.`assignmentsoffercollectionspersonalassignment` AS b
USING (
    SELECT
        case_account_for_search,
        -- Expire the assignment: if CAC_CREATED is older than 30 days,
        -- the record is stale -> write User as NULL in the target.
        CASE WHEN datediff(current_date(), CAC_CREATED) > 30 THEN NULL ELSE User END AS User
    FROM `p-cdh`.landed.qca_personal_assignment_8702 a
    join max_dt b on a.BusinessDate=b.businessdate
) AS a
    ON a.case_account_for_search = b.account_number
    AND b.businessdate = date_format(getdate(), 'yyyyMMdd')

WHEN MATCHED THEN
  UPDATE SET
    `User` = a.User,
    IngestionTime = current_timestamp()

WHEN NOT MATCHED BY TARGET THEN
  INSERT (account_number, `User`, businessdate, IngestionTime)
  VALUES (a.case_account_for_search, a.User, date_format(getdate(), 'yyyyMMdd'), current_timestamp());
  ''')